In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__results__.html
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/__output__.json
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/cleaned_HVA_dataset.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.xlsx
/kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/custom.css


In [3]:
# ============================================================
# Cross-model generalization experiments
#
# Train:
#   human + one AI source
#
# Test:
#   human + a different AI source
#
# No text preprocessing
# Group split by filename to prevent leakage
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")

# ============================================================
# 1) Load already-preprocessed long-format dataset
# ============================================================

DATA_PATH = (
    "/kaggle/input/notebooks/aabdollahii/"
    "8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv"
)

print("Dataset exists:", os.path.exists(DATA_PATH))
print("Dataset path:", DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nOriginal shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

# ============================================================
# 2) Validate columns
# ============================================================

required_columns = ["filename", "source", "content"]
missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Do not preprocess content.
# Only remove rows with missing required values.
df = df.dropna(
    subset=["filename", "source", "content"]
).copy()

df["source"] = df["source"].astype(str).str.strip().str.lower()

valid_sources = ["human", "gpt", "grok", "qwen"]

df = df[
    df["source"].isin(valid_sources)
].reset_index(drop=True)

# Human = 0, AI = 1
df["label"] = np.where(
    df["source"] == "human",
    0,
    1
)

print("\nShape after validation:", df.shape)

print("\nSource distribution:")
print(df["source"].value_counts())

print("\nLabel distribution:")
print(
    df["label"]
    .value_counts()
    .rename(index={0: "human", 1: "machine"})
)

# ============================================================
# 3) Keep complete filename groups
#
# A complete group should contain:
# human, gpt, grok, and qwen
#
# This ensures that the same filename can be used consistently
# for every train/test comparison.
# ============================================================

source_counts_per_file = (
    df.groupby("filename")["source"]
    .nunique()
)

complete_filenames = source_counts_per_file[
    source_counts_per_file == 4
].index

df_complete = df[
    df["filename"].isin(complete_filenames)
].reset_index(drop=True)

print("\nNumber of complete filenames:", len(complete_filenames))
print("Shape using complete filename groups:", df_complete.shape)

print("\nSource distribution after complete-group filtering:")
print(df_complete["source"].value_counts())

# Check whether a filename has more than one row for a source
duplicate_source_rows = (
    df_complete
    .groupby(["filename", "source"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_source_rows[
    duplicate_source_rows["count"] > 1
]

if len(duplicates) > 0:
    print(
        "\nWarning: Some filename/source combinations have "
        "multiple rows:"
    )
    display(duplicates.head())
else:
    print(
        "\nEach filename/source combination has exactly one row."
    )

# ============================================================
# 4) One grouped split used for every experiment
#
# This reserves the same filenames for testing in every run.
# Approximately:
#   80% filename groups -> training
#   20% filename groups -> testing
#
# Therefore, human records are not reused between train and test.
# ============================================================

all_filenames = df_complete["filename"].values
dummy_y = df_complete["label"].values

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_indices, test_indices = next(
    group_splitter.split(
        df_complete,
        dummy_y,
        groups=all_filenames
    )
)

train_filenames = set(
    df_complete.iloc[train_indices]["filename"]
)

test_filenames = set(
    df_complete.iloc[test_indices]["filename"]
)

filename_overlap = train_filenames.intersection(test_filenames)

print("\nTraining filename groups:", len(train_filenames))
print("Testing filename groups :", len(test_filenames))
print("Filename overlap        :", len(filename_overlap))

if len(filename_overlap) > 0:
    raise RuntimeError(
        "Filename leakage detected between train and test."
    )

# ============================================================
# 5) Define models
# ============================================================

def build_models():
    tfidf = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        lowercase=False
    )

    return {
        "TFIDF+MultinomialNB": Pipeline([
            ("tfidf", clone(tfidf)),
            ("classifier", MultinomialNB(alpha=1.0))
        ]),

        "TFIDF+LinearSVC": Pipeline([
            ("tfidf", clone(tfidf)),
            ("classifier", LinearSVC(
                C=1.0,
                random_state=42
            ))
        ]),

        "TFIDF+RandomForest": Pipeline([
            ("tfidf", clone(tfidf)),
            ("classifier", RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            ))
        ]),

        "TFIDF+KNN": Pipeline([
            ("tfidf", clone(tfidf)),
            ("classifier", KNeighborsClassifier(
                n_neighbors=15,
                metric="cosine"
            ))
        ])
    }

# ============================================================
# 6) Evaluation function
# ============================================================

def evaluate_experiment(
    train_data,
    test_data,
    train_ai_source,
    test_ai_source,
    model_name,
    model
):
    model.fit(
        train_data["content"].values,
        train_data["label"].values
    )

    y_true = test_data["label"].values
    y_pred = model.predict(test_data["content"].values)

    y_score = None

    # Probability score for NB, RF, and KNN
    if hasattr(model, "predict_proba"):
        try:
            probabilities = model.predict_proba(
                test_data["content"].values
            )

            if probabilities.shape[1] == 2:
                y_score = probabilities[:, 1]
        except Exception:
            y_score = None

    # Decision score for LinearSVC
    if y_score is None and hasattr(model, "decision_function"):
        try:
            decision_scores = model.decision_function(
                test_data["content"].values
            )

            if np.ndim(decision_scores) == 1:
                y_score = decision_scores
        except Exception:
            y_score = None

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            pos_label=1,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    roc_auc = np.nan

    if y_score is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_score)
        except Exception:
            roc_auc = np.nan

    print("\n" + "=" * 80)
    print(
        f"Train: human + {train_ai_source} | "
        f"Test: human + {test_ai_source}"
    )
    print(f"Model: {model_name}")
    print("=" * 80)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    if not np.isnan(roc_auc):
        print(f"ROC-AUC  : {roc_auc:.4f}")

    print("\nConfusion matrix:")
    display(pd.DataFrame(
        cm,
        index=["true_human", "true_machine"],
        columns=["pred_human", "pred_machine"]
    ))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["human", "machine"],
            zero_division=0
        )
    )

    result = {
        "train_ai_source": train_ai_source,
        "test_ai_source": test_ai_source,
        "model": model_name,
        "train_rows": len(train_data),
        "test_rows": len(test_data),
        "train_human_rows": int(
            (train_data["source"] == "human").sum()
        ),
        "train_ai_rows": int(
            (train_data["source"] == train_ai_source).sum()
        ),
        "test_human_rows": int(
            (test_data["source"] == "human").sum()
        ),
        "test_ai_rows": int(
            (test_data["source"] == test_ai_source).sum()
        ),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1]
    }

    prediction_data = test_data[
        ["filename", "source", "content", "label"]
    ].copy()

    prediction_data["y_true"] = y_true
    prediction_data["y_pred"] = y_pred

    if y_score is not None:
        prediction_data["score_machine"] = y_score

    return result, prediction_data, model

# ============================================================
# 7) Run all six train/test directions
# ============================================================

ai_sources = ["gpt", "grok", "qwen"]

all_results = []
all_predictions = {}
all_trained_models = {}
model_errors = []

for train_ai_source in ai_sources:

    # Training contains human + one AI source.
    train_data = df_complete[
        (
            df_complete["filename"].isin(train_filenames)
        )
        &
        (
            df_complete["source"].isin(
                ["human", train_ai_source]
            )
        )
    ].copy().reset_index(drop=True)

    for test_ai_source in ai_sources:

        if test_ai_source == train_ai_source:
            continue

        # Testing contains human + a different AI source.
        test_data = df_complete[
            (
                df_complete["filename"].isin(test_filenames)
            )
            &
            (
                df_complete["source"].isin(
                    ["human", test_ai_source]
                )
            )
        ].copy().reset_index(drop=True)

        print("\n\n")
        print("#" * 80)
        print(
            f"Experiment: train human + {train_ai_source}, "
            f"test human + {test_ai_source}"
        )
        print("Training shape:", train_data.shape)
        print("Testing shape :", test_data.shape)
        print("#" * 80)

        models = build_models()

        for model_name, model in models.items():

            try:
                result, prediction_data, trained_model = (
                    evaluate_experiment(
                        train_data=train_data,
                        test_data=test_data,
                        train_ai_source=train_ai_source,
                        test_ai_source=test_ai_source,
                        model_name=model_name,
                        model=model
                    )
                )

                all_results.append(result)

                experiment_name = (
                    f"train_{train_ai_source}"
                    f"_test_{test_ai_source}"
                    f"_{model_name}"
                )

                all_predictions[experiment_name] = prediction_data
                all_trained_models[experiment_name] = trained_model

            except Exception as error:
                print(
                    f"Error in {train_ai_source} -> "
                    f"{test_ai_source} with {model_name}:"
                )
                print(error)

                model_errors.append({
                    "train_ai_source": train_ai_source,
                    "test_ai_source": test_ai_source,
                    "model": model_name,
                    "error": str(error)
                })

# ============================================================
# 8) Compare all results
# ============================================================

results_df = pd.DataFrame(all_results)

if len(results_df) > 0:
    results_df = results_df.sort_values(
        by=["test_ai_source", "f1"],
        ascending=[True, False]
    ).reset_index(drop=True)

print("\nFinal cross-model results:")
display(results_df)

# Compact comparison table
if len(results_df) > 0:
    print("\nF1-score comparison:")
    display(
        results_df.pivot_table(
            index=["train_ai_source", "test_ai_source"],
            columns="model",
            values="f1"
        ).round(4)
    )

# ============================================================
# 9) Save all outputs
# ============================================================

results_path = (
    "/kaggle/working/"
    "HVA_cross_model_generalization_results.csv"
)

results_df.to_csv(
    results_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved results:", results_path)

for experiment_name, prediction_data in all_predictions.items():

    safe_name = (
        experiment_name
        .replace("+", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    prediction_path = (
        f"/kaggle/working/"
        f"HVA_{safe_name}_predictions.csv"
    )

    prediction_data.to_csv(
        prediction_path,
        index=False,
        encoding="utf-8-sig"
    )

print("Saved prediction files:", len(all_predictions))

if len(model_errors) > 0:
    errors_path = (
        "/kaggle/working/"
        "HVA_cross_model_generalization_errors.csv"
    )

    pd.DataFrame(model_errors).to_csv(
        errors_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved errors:", errors_path)


Dataset exists: True
Dataset path: /kaggle/input/notebooks/aabdollahii/8-ml-solution-final-data-analysis/HVA_long_preprocessed.csv

Original shape: (5647, 7)
Columns: ['year', 'filename', 'word_count', 'content', 'source', 'label', 'n_words']


,year,filename,word_count,content,source,label,n_words
0,2003,HAM2-811011-027.ham,136,آغاز عملیات اجرایی سد جدید بر روی رودخانه کارو...,gpt,1,96
1,2003,HAM2-811011-027.ham,136,عملیات اجرایی بدنه و سرریز سد کارون ۴ که بلندت...,grok,1,92
2,2003,HAM2-811011-027.ham,136,آغاز ساخت بدنه و سرریز بلندترین سد کشور عملیات...,human,0,137
3,2003,HAM2-811011-027.ham,136,آغاز فازهای کلیدی ساخت بزرگ‌ترین سازه آبی کشور...,qwen,1,209
4,2003,HAM2-811014-088.ham,53,گزارش تازه آب ذخیره‌شده در سدهای تهران نشان می...,gpt,1,92



Shape after validation: (5647, 7)

Source distribution:
source
grok     1519
human    1519
qwen     1519
gpt      1090
Name: count, dtype: int64

Label distribution:
label
machine    4128
human      1519
Name: count, dtype: int64

Number of complete filenames: 1090
Shape using complete filename groups: (4360, 7)

Source distribution after complete-group filtering:
source
gpt      1090
grok     1090
human    1090
qwen     1090
Name: count, dtype: int64

Each filename/source combination has exactly one row.

Training filename groups: 872
Testing filename groups : 218
Filename overlap        : 0



################################################################################
Experiment: train human + gpt, test human + grok
Training shape: (1744, 7)
Testing shape : (436, 7)
################################################################################

Train: human + gpt | Test: human + grok
Model: TFIDF+MultinomialNB
Accuracy : 0.6972
Precision: 0.8413
Recall   : 0.4862
F1-score : 0

,pred_human,pred_machine
true_human,198,20
true_machine,112,106



Classification report:
              precision    recall  f1-score   support

       human       0.64      0.91      0.75       218
     machine       0.84      0.49      0.62       218

    accuracy                           0.70       436
   macro avg       0.74      0.70      0.68       436
weighted avg       0.74      0.70      0.68       436


Train: human + gpt | Test: human + grok
Model: TFIDF+LinearSVC
Accuracy : 0.6193
Precision: 0.8939
Recall   : 0.2706
F1-score : 0.4155
ROC-AUC  : 0.8533

Confusion matrix:


...


Classification report:
              precision    recall  f1-score   support

       human       0.52      0.86      0.65       218
     machine       0.60      0.21      0.31       218

    accuracy                           0.53       436
   macro avg       0.56      0.53      0.48       436
weighted avg       0.56      0.53      0.48       436


Final cross-model results:


,train_ai_source,test_ai_source,model,train_rows,test_rows,train_human_rows,train_ai_rows,test_human_rows,test_ai_rows,accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp
0,qwen,gpt,TFIDF+MultinomialNB,1744,436,872,872,218,218,0.795872,0.814634,0.766055,0.789598,0.889235,180,38,51,167
1,grok,gpt,TFIDF+RandomForest,1744,436,872,872,218,218,0.763761,0.807487,0.692661,0.745679,0.865636,182,36,67,151
2,qwen,gpt,TFIDF+KNN,1744,436,872,872,218,218,0.724771,0.806250,0.591743,0.682540,0.815367,187,31,89,129
3,qwen,gpt,TFIDF+LinearSVC,1744,436,872,872,218,218,0.616972,0.947368,0.247706,0.392727,0.904680,215,3,164,54
4,grok,gpt,TFIDF+LinearSVC,1744,436,872,872,218,218,0.564220,0.868421,0.151376,0.257812,0.817208,213,5,185,33
5,grok,gpt,TFIDF+KNN,1744,436,872,872,218,218,0.516055,0.620690,0.082569,0.145749,0.624137,207,11,200,18
6,qwen,gpt,TFIDF+RandomForest,1744,436,872,872,218,218,0.534404,0.894737,0.077982,0.143460,0.788307,216,2,201,17
7,grok,gpt,TFIDF+MultinomialNB,1744,436,872,872,218,218,0.500000,0.000000,0.000000,0.000000,0.825856,218,0,218,0
8,gpt,grok,TFIDF+MultinomialNB,1744,436,872,872,218,218,0.697248,0.841270,0.486239,0.616279,0.823142,198,20,112,106
9,gpt,grok,TFIDF+KNN,1744,436,872,872,218,218,0.607798,0.711712,0.362385,0.480243,0.667105,186,32,139,79



F1-score comparison:


model                           TFIDF+KNN  TFIDF+LinearSVC  \
train_ai_source test_ai_source                               
gpt             grok               0.4802           0.4155   
                qwen               0.7593           0.7936   
grok            gpt                0.1457           0.2578   
                qwen               0.0343           0.0178   
qwen            gpt                0.6825           0.3927   
                grok               0.3119           0.0268   

model                           TFIDF+MultinomialNB  TFIDF+RandomForest  
train_ai_source test_ai_source                                           
gpt             grok                         0.6163              0.3121  
                qwen                         0.8879              0.4036  
grok            gpt                          0.0000              0.7457  
                qwen                         0.0000              0.0310  
qwen            gpt                          0.7896              0.1435  
                grok                         0.4625              0.0000


Saved results: /kaggle/working/HVA_cross_model_generalization_results.csv
Saved prediction files: 24
